In [25]:
library(tidyverse)
library(MatchIt)
library(broom)
library(tibble)

In [3]:
file_path <- "/home/bingkzhao2/patent_pwords/Bingkun/Data/appeal_reg_data.csv"
df <- read.csv(file_path, stringsAsFactors = FALSE)

In [4]:
df_clean <- df %>%
  mutate(

    hype_quintile = ntile(frac_hype_words, 5),

    team_gender_n = as.factor(team_gender),
    attorney_gender_n = as.factor(attorney_gender),
    examiner_gender_n = as.factor(examiner_gender),
    cpc_class_n = as.factor(cpc_class),
    cpc_section_n = as.factor(cpc_section),
    applicant_entity_n = as.factor(applicant_entity),
    business_entity_status_n = as.factor(business_entity_status),
    year = as.factor(year), 
    
    num_words_log = log(num_words + 1),
    num_prior_apps_log = log(num_prior_apps + 1),
    num_prior_patents_log = log(num_prior_patents + 1),
    attorney_num_prior_apps_log = log(attorney_num_prior_apps + 1),
    examiner_num_prior_apps_log = log(examiner_num_prior_apps + 1),
    attorney_num_prior_patents_log = log(attorney_num_prior_patents + 1),
    
    num_words_5cate = as.factor(ntile(num_words, 5)),
    examiner_num_prior_apps_5cate = as.factor(ntile(examiner_num_prior_apps, 5)),

    novelty_min_qnorm = percent_rank(novelty_tail_10)
  )

In [22]:
psm_covariates_formula_appeal <- as.formula("is_Q5 ~ flesch_reading_ease + concreteness + 
                                     novelty_min_qnorm + num_cpc_code + num_inventors + 
                                     num_words_5cate + 
                                     num_prior_apps_log + num_prior_patents_log + 
                                     attorney_num_prior_apps_log + 
                                     team_gender_n + attorney_gender_n + 
                                     year + cpc_class_n + applicant_entity_n + business_entity_status_n")

## PSM

In [23]:
results_table <- data.frame()
comparison_groups <- c(1, 2, 3, 4)

for (comp_group in comparison_groups) {
  
  message(paste0("--------------------------------------------------"))
  message(paste0("Computing: Q", comp_group, " (Control) vs Q5 (Treated)..."))
  
  df_sub <- df_clean %>%
    filter(hype_quintile %in% c(comp_group, 5)) %>%
    mutate(
      is_Q5 = ifelse(hype_quintile == 5, 1, 0)
    )
  
  m.out <- matchit(psm_covariates_formula_appeal,
                   data = df_sub,
                   method = "nearest", 
                   ratio = 1,
                   replace = FALSE)
  
  df_matched <- match.data(m.out)
  

  q5_outcomes <- df_matched$appeal_decision[df_matched$is_Q5 == 1]
  control_outcomes <- df_matched$appeal_decision[df_matched$is_Q5 == 0]  
  t_res <- t.test(x = q5_outcomes, y = control_outcomes)  
  diff_val <- t_res$estimate[2] - t_res$estimate[1]
  ci_lower_flipped <- t_res$conf.int[2] * -1
  ci_upper_flipped <- t_res$conf.int[1] * -1

  tidy_res <- data.frame(
    term = "is_Q5",
    estimate = diff_val,              
    std.error = NA,                   
    p.value = t_res$p.value,          
    conf.low = ci_lower_flipped,      
    conf.high = ci_upper_flipped,     
    
    comparison = paste0("Q", comp_group, " vs Q5"),
    group_id = comp_group,
    mean_Q5 = t_res$estimate[1],
    mean_Control = t_res$estimate[2]
  )
  
  results_table <- bind_rows(results_table, tidy_res)  
}

print(results_table %>% select(comparison, estimate, p.value, conf.low, conf.high))

--------------------------------------------------

Computing: Q1 (Control) vs Q5 (Treated)...

--------------------------------------------------

Computing: Q2 (Control) vs Q5 (Treated)...

--------------------------------------------------

Computing: Q3 (Control) vs Q5 (Treated)...

--------------------------------------------------

Computing: Q4 (Control) vs Q5 (Treated)...



              comparison   estimate      p.value     conf.low  conf.high
mean of y...1   Q1 vs Q5 0.05333941 1.481202e-07  0.033458390 0.07322043
mean of y...2   Q2 vs Q5 0.04422156 1.246273e-05  0.024393597 0.06404953
mean of y...3   Q3 vs Q5 0.02165489 3.103824e-02  0.001974756 0.04133502
mean of y...4   Q4 vs Q5 0.00661044 5.078642e-01 -0.012957857 0.02617874


## BALANCE CHECK

In [27]:
balance_table <- data.frame()
comparison_groups <- c(1, 2, 3, 4)

for (comp_group in comparison_groups) {
  
  message(paste0("--------------------------------------------------"))
  message(paste0("Computing: Q", comp_group, " (Control) vs Q5 (Treated)..."))

  df_sub <- df_clean %>%
    filter(hype_quintile %in% c(comp_group, 5)) %>%
    mutate(
      is_Q5 = ifelse(hype_quintile == 5, 1, 0)
    )
  
  # Notice: using your specific psm_covariates_formula_appeal
  m.out <- matchit(psm_covariates_formula_appeal,
                   data = df_sub,
                   method = "nearest", 
                   distance = "glm",  
                   caliper = 0.05,    
                   ratio = 1,
                   replace = FALSE)
  
  balance_summary <- summary(m.out, un = TRUE) 
  
  tidy_balance <- as.data.frame(balance_summary$sum.matched) %>%
    rownames_to_column(var = "covariate") %>%               # Bring row names into a proper column
    select(covariate, matched_SMD = `Std. Mean Diff.`) %>%  # Select only the covariate name and SMD
    mutate(
      comparison = paste0("Q", comp_group, " vs Q5"),       # Add a column for the comparison group
      group_id = comp_group
    )
  
  # Bind the rows to the master balance_table
  balance_table <- bind_rows(balance_table, tidy_balance)
}

# Print the final compiled tidy table
message("\n=== FINAL TIDY BALANCE TABLE ===")
print(balance_table)

--------------------------------------------------

Computing: Q1 (Control) vs Q5 (Treated)...

--------------------------------------------------

Computing: Q2 (Control) vs Q5 (Treated)...

--------------------------------------------------

Computing: Q3 (Control) vs Q5 (Treated)...

--------------------------------------------------

Computing: Q4 (Control) vs Q5 (Treated)...


=== FINAL TIDY BALANCE TABLE ===



                               covariate   matched_SMD comparison group_id
1                               distance  0.0395320893   Q1 vs Q5        1
2                    flesch_reading_ease -0.0030727474   Q1 vs Q5        1
3                           concreteness -0.0126049537   Q1 vs Q5        1
4                      novelty_min_qnorm  0.0123553927   Q1 vs Q5        1
5                           num_cpc_code -0.0025120230   Q1 vs Q5        1
6                          num_inventors -0.0131619850   Q1 vs Q5        1
7                       num_words_5cate1  0.0068373193   Q1 vs Q5        1
8                       num_words_5cate2 -0.0033808808   Q1 vs Q5        1
9                       num_words_5cate3 -0.0113740294   Q1 vs Q5        1
10                      num_words_5cate4 -0.0037617674   Q1 vs Q5        1
11                      num_words_5cate5  0.0118038291   Q1 vs Q5        1
12                    num_prior_apps_log -0.0010338710   Q1 vs Q5        1
13                 num_pr